# Quickstart: computing climate indicators with earthkit-climate

This tutorial provides a short, hands-on introduction to computing climate indicators using .
It focuses on core workflows with minimal end-to-end examples, relying on sensible defaults wherever possible.

In the  ecosystem, data retrieved via  (as  or  objects) can be passed directly to  indicators. Every indicator function is decorated with , which handles format conversion behind the scenes without requiring manual  calls.

We will cover three progressively more complex examples:
1. **Single input variable**: Computing a simple threshold indicator (number of hot days).
2. **Climatology-based indicator**: Computing an indicator that uses a baseline reference climatology.
3. **Percentile-based indicator**: Computing an indicator using percentile thresholds.


In [1]:
import numpy as np
import pandas as pd
import xarray as xr

import earthkit.climate as ekc

## 1. Simple indicator with a single input variable

We begin by generating synthetic daily maximum temperature data () over a multi-year period (in Kelvin).
We then compute the **number of hot days** per year where maximum temperature exceeds 30°C (303.15 K).


In [2]:
# Create synthetic daily maximum temperature data (2020-2023)
dates = pd.date_range("2020-01-01", "2023-12-31", freq="D")
np.random.seed(42)
temp_values = 285.0 + 15.0 * np.sin(2 * np.pi * dates.dayofyear / 365.25) + np.random.normal(0, 3, len(dates))

tasmax = xr.DataArray(
    temp_values,
    coords={"time": dates},
    dims=["time"],
    name="tasmax",
    attrs={"units": "K", "standard_name": "air_temperature"},
)

# Compute annual number of hot days (> 30 deg C / 303.15 K)
hot_days = ekc.indicators.tx_days_above(tasmax, thresh="30 degC", freq="YS")
print("Annual hot days:")
print(hot_days)

Annual hot days:
<xarray.DataArray 'tx_days_above' (time: 4)> Size: 32B
array([6., 7., 9., 9.])
Coordinates:
  * time     (time) datetime64[us] 32B 2020-01-01 2021-01-01 ... 2023-01-01
Attributes:
    units:          days
    standard_name:  number_of_days_with_air_temperature_above_threshold
    cell_methods:    time: sum over days
    history:        [2026-08-28 14:03:17] tx_days_above: TX_DAYS_ABOVE(tasmax...
    long_name:      The number of days with maximum temperature above 30 degc
    description:    Annual number of days where daily maximum temperature exc...


/home/cuadradot/predictia-projects/git/c3s-indices/earthkit-climate/.venv/lib/python3.12/site-packages/xclim/core/cfchecks.py:77: UserWarning: Variable does not have a `cell_methods` attribute.
  _check_cell_methods(getattr(vardata, "cell_methods", None), data["cell_methods"])


## 2. Indicator using a baseline climatology

Many climate indicators evaluate anomalies relative to a baseline climatology period (e.g. 1991–2020).
In the earthkit ecosystem, baseline climatologies are prepared using [earthkit-transforms](https://earthkit-transforms.readthedocs.io/en/latest/concepts/climatology.html).

Here we create a synthetic 30-year reference period to compute a mean daily climatology, and then evaluate temperature anomalies relative to this baseline.


In [3]:
# 30-year reference daily temperature dataset (1991-2020)
ref_dates = pd.date_range("1991-01-01", "2020-12-31", freq="D")
ref_temp = 285.0 + 15.0 * np.sin(2 * np.pi * ref_dates.dayofyear / 365.25) + np.random.normal(0, 2, len(ref_dates))

tasmax_ref = xr.DataArray(
    ref_temp,
    coords={"time": ref_dates},
    dims=["time"],
    name="tasmax",
    attrs={"units": "K", "standard_name": "air_temperature"},
)

# Daily mean climatology across the 30-year reference period
daily_climatology = tasmax_ref.groupby("time.dayofyear").mean("time")
print("Daily climatology (first 5 days of year):")
print(daily_climatology.head(5))

# Compute daily temperature anomalies relative to the baseline climatology
anomalies = tasmax.groupby("time.dayofyear") - daily_climatology
print("Daily temperature anomalies (first 5 days):")
print(anomalies.head(5))

Daily climatology (first 5 days of year):
<xarray.DataArray 'tasmax' (dayofyear: 5)> Size: 40B
array([285.36765602, 285.64847558, 286.40183678, 285.91473316,
       285.28236837])
Coordinates:
  * dayofyear  (dayofyear) int64 40B 1 2 3 4 5
Attributes:
    units:          K
    standard_name:  air_temperature
Daily temperature anomalies (first 5 days):
<xarray.DataArray 'tasmax' (time: 5)> Size: 40B
array([ 1.38051007, -0.54729758,  1.31499434,  4.68568753,  0.30376307])
Coordinates:
  * time       (time) datetime64[us] 40B 2020-01-01 2020-01-02 ... 2020-01-05
    dayofyear  (time) int64 40B 1 2 3 4 5
Attributes:
    units:          K
    standard_name:  air_temperature


## 3. Indicator using percentiles

Percentile-based indices (such as , the fraction of days where maximum temperature exceeds the 90th calendar day percentile) require rolling daily percentile thresholds.

> **Note on earthkit-transforms**: While  provides general percentile calculations, ongoing work is extending  to natively support rolling daily percentile window calculations. Currently,  provides helper utilities () or integrates directly with  percentile estimation engines.


In [4]:
# Calculate 90th percentile threshold across rolling 5-day windows for day of year
per90 = ekc.utils.climatology.rolling_percentiles(tasmax_ref, p=90, window_width=5)

# Calculate warm days index (tx90p) for a target year (2023)
tx90p_index = ekc.indicators.tx90p(tasmax, tasmax_per=per90, freq="YS")
print("Annual percentage of warm days (tx90p):")
print(tx90p_index)

Annual percentage of warm days (tx90p):
<xarray.DataArray 'tx90p' (time: 4, percentile: 1)> Size: 32B
array([[61.],
       [69.],
       [80.],
       [80.]])
Coordinates:
  * time        (time) datetime64[us] 32B 2020-01-01 2021-01-01 ... 2023-01-01
  * percentile  (percentile) int64 8B 90
Attributes:
    units:               days
    standard_name:       days_with_air_temperature_above_threshold
    climatology_bounds:  ['1991-01-01', '2020-12-31']
    window:              5
    alpha:               0.3333333333333333
    beta:                0.3333333333333333
    history:             [2026-08-28 14:03:18] tx90p: TX90P(tasmax=tasmax, ta...
    cell_methods:         time: sum over days
    long_name:           Number of days with maximum temperature above the 90...
    description:         Annual number of days with maximum temperature above...


/home/cuadradot/predictia-projects/git/c3s-indices/earthkit-climate/.venv/lib/python3.12/site-packages/xclim/core/cfchecks.py:77: UserWarning: Variable does not have a `cell_methods` attribute.
  _check_cell_methods(getattr(vardata, "cell_methods", None), data["cell_methods"])
